# pyAVS quickstart: MEG + eye-tracking on the AVS dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KietzmannLab/pyavs/blob/main/examples/pyavs_colab_quickstart.ipynb)

A ~15 minute, hands-on first contact with the **AVS dataset** (MEG + eye-tracking + MRI, 5
participants viewing MS-COCO natural scenes) via the **pyAVS** Python package. Using one
subject/session, this notebook covers:

1. Loading raw MEG and eye-tracking data
2. Loading a scene image together with the fixations made on it
3. A flagship sensor-space comparison: MEG responses to fixations on **dogs** vs **airplanes**

This is a quick "raw ingredients + one precomputed derivative" tour, not a rerun of the full
preprocessing pipeline (Maxwell filtering, ICA, epoching) — for that, and for a broader tour
of the package, see **Where to go next** at the end.

> **Scope note:** the dog-vs-airplane MEG contrast in Section 5 is computed for a single
> subject and is meant to illustrate the tooling, not as a statistical claim about object-category
> representation in MEG — no significance testing is performed here.

## 1. Setup

In [3]:
%pip install -q git+https://github.com/KietzmannLab/pyavs.git

You should consider upgrading via the '/Users/atlas/Documents/Documents_atlas/PhD/code/pyavs_conversion/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 2. Get the data

This notebook uses **subject 1, session 1** only, downloaded on demand from the public
`avs-public` dataset on AWS ([Open Data Sponsorship
Program](https://registry.opendata.aws/), bucket `kietzmannlab-avs`, `us-west-2`). The
bucket is public/anonymous-read, so no AWS account or credentials are needed — files are
fetched with a plain HTTPS GET.

In [4]:
import pathlib

import requests

S3_BUCKET = "kietzmannlab-avs"
S3_REGION = "us-west-2"
S3_BASE_URL = f"https://{S3_BUCKET}.s3.{S3_REGION}.amazonaws.com"

DATA_DIR = pathlib.Path("avs_quickstart_data")


def s3_download(rel_path, dest_root=DATA_DIR, required=True):
    '''Download one avs-public-relative path from the public S3 bucket, atomically,
    skipping files already on disk. Returns None for a missing (still-uploading) key
    when required=False, otherwise raises.'''
    dest = pathlib.Path(dest_root) / rel_path
    if dest.exists():
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.parent / (dest.name + ".part")
    with requests.get(f"{S3_BASE_URL}/{rel_path}", stream=True, timeout=1800) as resp:
        if resp.status_code == 404 and not required:
            return None
        resp.raise_for_status()
        with open(tmp, "wb") as fh:
            for chunk in resp.iter_content(1 << 20):
                fh.write(chunk)
    tmp.rename(dest)
    return dest


CORE_FILES = [
    "sub-01/ses-01/meg/as01a01.fif",                                                            # raw MEG, block 1
    "sub-01/ses-01/beh/as_exp_data_1_1_3_0.parquet",                                             # experiment log
    "derivatives/pyavs/sub-01/ses-01/eyetrack/as_s1_el_events.parquet",                          # cleaned ET events
    "derivatives/pyavs/sub-01/ses-01/eyetrack/as_s1_el_msgs.parquet",                            # cleaned ET messages
    "derivatives/pyavs/sub-01/ses-01/epochs/sub-01_ses-01_task-avs_fixation_scene_epochs.h5",    # fixation-locked MEG epochs (~2 GB)
    "derivatives/pyavs/sub-01/ses-01/epochs/sub-01_ses-01_fixation_metadata.parquet",            # per-fixation metadata (incl. object_label)
]
for rel in CORE_FILES:
    print(f"  downloading {rel} ...")
    s3_download(rel)
print("done")

  downloading sub-01/ses-01/meg/as01a01.fif ...
  downloading sub-01/ses-01/beh/as_exp_data_1_1_3_0.parquet ...
  downloading derivatives/pyavs/sub-01/ses-01/eyetrack/as_s1_el_events.parquet ...
  downloading derivatives/pyavs/sub-01/ses-01/eyetrack/as_s1_el_msgs.parquet ...
  downloading derivatives/pyavs/sub-01/ses-01/epochs/sub-01_ses-01_task-avs_fixation_scene_epochs.h5 ...
  downloading derivatives/pyavs/sub-01/ses-01/epochs/sub-01_ses-01_fixation_metadata.parquet ...
done


In [5]:
import pyavs

pyavs.set_data_path(str(DATA_DIR))

pyavs.check_data_availability(subject_id=1, session=1)

[2026-08-20 08:41:15] pyavs.logging - INFO - Logging configured: level=INFO, console=True, file=False
[2026-08-20 08:41:15] pyavs - WARNING - Logging already configured. Reconfiguring...
[2026-08-20 08:41:15] pyavs.logging - INFO - Logging configured: level=INFO, console=True, file=False


ModuleNotFoundError: No module named 'torch'

## 3. Raw ingredients: MEG, eye events, and a scene image

Start with one MEG recording block and a quick look at the signal.

In [ ]:
raw = pyavs.load_meg_raw(subject_id=1, session=1, run=1, data_path=str(DATA_DIR), preload=True, verbose=True)
raw

In [ ]:
raw.compute_psd(fmax=100, picks="meg").plot();

Now the eye-tracking side: cleaned fixation/saccade events for the same subject/session,
enriched with trial/block/scene context (which scene, which block, time within the trial,
and so on). Object labels for what was fixated (e.g. "dog") aren't part of this
general-purpose events table — Section 5 pulls those separately from the fixation-locked
epochs' own metadata.

In [ ]:
explog_df, events_df = pyavs.load_and_enrich_eye_events(subjects=[1], sessions=[1], data_path=str(DATA_DIR))

fixations = events_df[(events_df["type"] == "fixation") & (events_df["recording"] == "scene")]
print(f"{len(fixations)} scene-viewing fixations, on {fixations['sceneID'].nunique()} distinct scenes")
fixations.head()

## 4. Eye movements on the scene

`EyeTrackingPlotter` overlays a subject's fixations directly on the COCO scene image they
viewed. `plot_scene` silently skips scenes the subject never actually viewed, so `scene_id`
below is pulled from the loaded fixation data rather than hardcoded.

In [ ]:
from pyavs.config.config import PyAVSConfig
from pyavs.visualization.events_on_scene import EyeTrackingPlotter

plotter = EyeTrackingPlotter(subjects=1, sessions=1, config=PyAVSConfig(), data_path=str(DATA_DIR))
scene_id = int(plotter.df["sceneID"].iloc[0])

**Note on image availability:** the full public release ships scene images directly under
`stimuli/images/`, but the S3 upload is proceeding incrementally, so a given viewed scene's
image may not be up yet. The cell below checks whether *this subject's* viewed scenes have an
image on S3, and swaps in a different viewed scene if not.

In [ ]:
candidates = plotter.df["sceneID"].dropna().unique().astype(int).tolist()
for candidate in candidates:
    if s3_download(f"stimuli/images/{candidate:012d}_MEG_size.jpg", required=False) is not None:
        scene_id = candidate
        break
else:
    raise RuntimeError("None of this subject's viewed scenes have an image on S3 yet -- try again later.")

plotter.plot_scene(scene_id, subject=1)

## 5. Flagship: MEG responses to dog vs. airplane fixations

The precomputed `fixation_scene` epochs ship as plain `grad`/`mag` arrays with no channel
identity attached, so they can't directly drive an MNE topomap out of the box. Every AVS
session uses the same fixed 306-channel Neuromag/MEGIN system, though, so we can borrow the
channel layout and digitization from the raw block already loaded in Section 3 and attach it
to the epoch arrays to build a proper `mne.EpochsArray`.

In [ ]:
import numpy as np
import mne

from pyavs.io.read import load_epochs_h5, load_metadata_csv

data_dict, _, attrs = load_epochs_h5(
    subject_id=1, session=1, event_type="fixation_scene", data_path=str(DATA_DIR)
)
metadata = load_metadata_csv(subject_id=1, session=1, event_type="fixation_scene", data_path=str(DATA_DIR))

grad_picks = mne.pick_types(raw.info, meg="grad")
mag_picks = mne.pick_types(raw.info, meg="mag")
epochs_info = mne.pick_info(raw.info, np.concatenate([grad_picks, mag_picks]))
with epochs_info._unlock():
    epochs_info["sfreq"] = attrs["hz"]  # h5's native rate (500 Hz); raw's own sfreq (1000 Hz) doesn't apply here

epochs_data = np.concatenate([data_dict["grad"], data_dict["mag"]], axis=1)  # (n_epochs, 306, n_times)

n_epochs = epochs_data.shape[0]
sfreq = attrs["hz"]
events = np.column_stack([
    np.arange(n_epochs) * int(sfreq),
    np.zeros(n_epochs, dtype=int),
    np.ones(n_epochs, dtype=int),
])
epochs = mne.EpochsArray(
    epochs_data, epochs_info, events=events, tmin=attrs["times"][0],
    event_id={"fixation": 1}, verbose=False,
)
epochs.metadata = metadata

n_dog, n_airplane = (metadata["object_label"] == "dog").sum(), (metadata["object_label"] == "airplane").sum()
print(f"dog: {n_dog} fixations, airplane: {n_airplane} fixations (of {n_epochs} total)")

Time series (mean over magnetometers):

In [ ]:
dog_evoked = epochs['object_label == "dog"'].average()
airplane_evoked = epochs['object_label == "airplane"'].average()

mne.viz.plot_compare_evokeds(
    {"dog": dog_evoked, "airplane": airplane_evoked},
    picks="mag", combine="mean", title="Fixation-locked MEG response (n=1 subject, illustrative only)",
);

Sensor-space topography at a fixed latency after fixation onset:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, evoked, label in zip(axes, [dog_evoked, airplane_evoked], ["dog", "airplane"]):
    evoked.plot_topomap(times=0.15, ch_type="mag", axes=ax, colorbar=False, show=False)
    ax.set_title(label)
fig.suptitle("Fixation-locked MEG topography, 150 ms post-fixation-onset (mag)")
plt.show()

## 6. Where to go next

This notebook only scratched the surface: raw data plus one precomputed derivative, no
preprocessing. For more:

- **Full MEG + eye-tracking preprocessing walkthrough** (Maxwell filtering, ICA, event
  alignment, epoching) via `AVSComposer`:
  [`docs/tutorials/meg_eye_workflow.rst`](https://github.com/KietzmannLab/pyavs/blob/main/docs/tutorials/meg_eye_workflow.rst)
- **More worked examples** (object detection, source reconstruction, population codes, config
  reproducibility): [`examples/`](https://github.com/KietzmannLab/pyavs/tree/main/examples)
- **Full analysis pipelines** used in the AVS paper (representational similarity analysis,
  ANN-to-MEG encoding, source-projected fixation ERFs, decoding):
  [`scripts/`](https://github.com/KietzmannLab/pyavs/tree/main/scripts)
- **Citing the dataset or pyAVS**:
  [`docs/reference/citation.rst`](https://github.com/KietzmannLab/pyavs/blob/main/docs/reference/citation.rst)

*(The full rendered documentation site is not live yet as of this writing — the links above
point at the source `.rst`/example files on GitHub in the meantime.)*